In [ ]:
!pip install kafka-python
!pip install google-api-python-client
!pip install boto3
!pip install mysql-connector-python
!pip install  pymysql



# Consumer Code

In [ ]:
from kafka import KafkaConsumer
import json
import mysql.connector
import boto3
import csv
import io
from datetime import datetime

# === AWS S3 Setup ===
AWS_ACCESS_KEY = ''
AWS_SECRET_KEY = ''
S3_BUCKET_NAME = ''
S3_FOLDER = 'youtube-data/'

s3_client = boto3.client(
    's3',
    aws_access_key_id=AWS_ACCESS_KEY,
    aws_secret_access_key=AWS_SECRET_KEY,
    region_name=''
)

# === MySQL Setup ===
db = mysql.connector.connect(
    host='',
    user='',
    password='',
    database=''
)
cursor = db.cursor()

insert_query = """
INSERT INTO youtube_videos (video_id, title, published_at, view_count, like_count, channel_title)
VALUES (%s, %s, %s, %s, %s, %s)
"""

# === Kafka Consumer Setup ===
consumer = KafkaConsumer(
    'youtube-topic',
    bootstrap_servers='',
    value_deserializer=lambda m: json.loads(m.decode('utf-8')),
    group_id='youtube-group',
    auto_offset_reset='earliest'
)

# === Consume Messages ===
for message in consumer:
    video = message.value
    try:
        # Sanitize/handle missing fields
        video_id = video.get('videoId', '')
        title = video.get('title', '')
        published_at = video.get('publishedAt', '')
        view_count = video.get('viewCount') or 0
        like_count = video.get('likeCount') or 0
        channel_title = video.get('channelTitle', '')

        # Insert into MySQL
        cursor.execute(insert_query, (
            video_id, title, published_at, view_count, like_count, channel_title
        ))
        db.commit()

        # Save to S3 with headers
        csv_buffer = io.StringIO()
        writer = csv.writer(csv_buffer)
        writer.writerow(['video_id', 'title', 'published_at', 'view_count', 'like_count', 'channel_title'])
        writer.writerow([video_id, title, published_at, view_count, like_count, channel_title])

        s3_key = f"{S3_FOLDER}video_{video_id}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
        s3_client.put_object(Bucket=S3_BUCKET_NAME, Key=s3_key, Body=csv_buffer.getvalue())

        print(f"✅ Stored to MySQL & ☁️ S3: {title}")
    except Exception as e:
        print(f"❌ Error processing video: {e}")

cursor.close()
db.close()


In [ ]:
import boto3
import pymysql
import pandas as pd
import os

# ---------- RDS Configuration ----------
rds_host = ''
rds_user = ''
rds_password = ''  # 🔒 REPLACE
rds_database = 'youtube_db'
rds_table = 'youtube_videos'

# ---------- S3 Configuration ----------
bucket_name = ''
s3_key = 'youtube-data/youtube_videos.csv'
local_path = '/tmp/youtube_videos.csv'

# ---------- Connect to RDS ----------
conn = pymysql.connect(
    host=rds_host,
    user=rds_user,
    password=rds_password,
    database=rds_database
)

# ---------- Fetch data into DataFrame ----------
query = f"SELECT * FROM {rds_table}"
df = pd.read_sql(query, conn)
conn.close()

# ---------- Clean & Export CSV ----------
# Replace newlines, ensure strings, avoid shifting
df = df.fillna('').astype(str)
df = df.applymap(lambda x: x.replace('\n', ' ').replace('\r', ' '))

# Write with quoting
df.to_csv(local_path, index=False, quoting=1)  # 1 = csv.QUOTE_ALL

# ---------- Upload to S3 ----------
s3 = boto3.client(
    's3',
    aws_access_key_id='',          # 🔒 Replace or remove if using EC2/IAM role
    aws_secret_access_key='',
    region_name=''
)

s3.upload_file(local_path, bucket_name, s3_key)
print(f"✅ Uploaded clean DataFrame CSV to s3://{bucket_name}/{s3_key}")
